In [ ]:
import os
import json
import math
import random
import tarfile
import logging
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm
from os.path import join, exists

from IPython.display import Audio, display

import librosa
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import snapshot_download, login as hf_login


import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel


load_dotenv()


HF_TOKEN = os.getenv("HF_TOKEN")


ROOT_DIR = Path("./").resolve()

hf_login(token=HF_TOKEN)

In [ ]:
logger = logging.getLogger("DCASE2026_Task6")
logger.setLevel(logging.INFO)

# Remove any existing handlers on this specific logger
logger.handlers.clear()

# Create handler + formatter and attach
handler = logging.StreamHandler()
handler.setFormatter(
    logging.Formatter(
        fmt="%(asctime)s.%(msecs)03d: [%(levelname)s] : [%(name)s] - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(handler)

# Prevent messages from propagating to Jupyter's root handler (avoids duplicate lines)
logger.propagate = False

logger.info("Hello world")

In [ ]:
# repo_id = "lighthouse-emnlp2024/Clotho-Moment_CLAP_features"
repo_id = "lighthouse-emnlp2024/Clotho-Moment"
# local_dir = "../data/clotho-moment"
local_dir = "./data/clotho-moment"

downloaded_path = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=local_dir,
    allow_patterns=["train/train-000.tar", "valid/valid-000.tar"],
)

In [ ]:
# tar_path = "./data/clotho-moment/train/train-000.tar"
tar_paths = [
    "./data/clotho-moment/train/train-000.tar",
    "./data/clotho-moment/valid/valid-000.tar",
]


for tar_path in tar_paths:
    extract_path = "./data/clotho-moment/"
    # Create directory if it doesn't exist
    os.makedirs(extract_path, exist_ok=True)

    with tarfile.open(tar_path) as tar:
        if "train" in tar_path:
            extract_path += "train/"
        elif "valid" in tar_path:
            extract_path += "valid/"
        else:
            pass
        tar.extractall(path=extract_path)
        print(f"Extracted all files to {extract_path}")

## Basic EDA

In [ ]:
DATA_DIR = ROOT_DIR / "data" / "clotho-moment"
# EXTRACTED_DATA_DIR  = DATA_DIR / "extracted"

TRAIN_DATASET_DIR = DATA_DIR / "train"
VALID_DATASET_DIR = DATA_DIR / "valid"

In [ ]:
# 1. Get all wav files (sorted for consistency)
wav_files = sorted(list(TRAIN_DATASET_DIR.glob("*.wav")))
# 2. Create a list of tuples: (wav_path, json_path)
# .with_suffix(".json") replaces .wav with .json
train_pairs = [
    (str(f), str(f.with_suffix(".json")))
    for f in wav_files
    if f.with_suffix(".json").exists()
]
# Print the first 2 to verify
print(f"Found {len(train_pairs)} matching pairs.")
for wav_file, json_file in train_pairs[:2]:
    print(f"Pair: ({wav_file}, {json_file})")

In [ ]:
# 1. Get all wav files (sorted for consistency)
val_wav_files = sorted(list(VALID_DATASET_DIR.glob("*.wav")))
# 2. Create a list of tuples: (wav_path, json_path)
# .with_suffix(".json") replaces .wav with .json
val_pairs = [
    (str(f), str(f.with_suffix(".json")))
    for f in val_wav_files
    if f.with_suffix(".json").exists()
]
# Print the first 2 to verify
print(f"Found {len(val_pairs)} matching pairs.")
for wav_file, json_file in val_pairs[:2]:
    print(f"Pair: ({wav_file}, {json_file})")

In [ ]:
def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(l.strip("\n")) for l in f.readlines()]

In [ ]:
dcase2026_task6_metadata_dir = ROOT_DIR / "dcase2026_task6_baseline" / "data"
dcase2026_clotho_train_json = load_jsonl(
    dcase2026_task6_metadata_dir / "clotho_moment_train_release.jsonl"
)

In [ ]:
# ---------------------------------------------------------------------------
# Build DataFrame with useful columns
# ---------------------------------------------------------------------------
df = pd.DataFrame(dcase2026_clotho_train_json)
# Clean video id (remove dot) to match file naming convention
df["vid_clean"] = df["vid"].str.replace(".", "")
# Add derived columns
df["query_len_chars"] = df["query"].apply(len)
# Token count (simple whitespace split)
df["query_len_tokens"] = df["query"].apply(lambda x: len(x.split()))
df["num_windows"] = df["relevant_windows"].apply(len)

print("=== Dataset Overview ===")
print(f"Total samples: {len(df)}")
print(f"Average duration (s): {df['duration'].mean():.2f}")
print(f"Average query length (chars): {df['query_len_chars'].mean():.2f}")
print(f"Average query length (tokens): {df['query_len_tokens'].mean():.2f}")
print(f"Average number of windows per sample: {df['num_windows'].mean():.2f}\n")

In [ ]:
# ---------------------------------------------------------------------------
# Visualizations
# ---------------------------------------------------------------------------
# 1. Duration distribution (Matplotlib + Seaborn)
plt.figure(figsize=(10, 6))
ax = sns.histplot(df["duration"], bins=30, kde=True, color="steelblue")
ax.set_title("Video Duration Distribution", fontsize=14, weight="bold")
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Count")
plt.show()

# 2. Query length (characters) distribution
plt.figure(figsize=(10, 6))
ax = sns.histplot(df["query_len_chars"], bins=30, kde=True, color="coral")
ax.set_title("Query Text Length (Characters)", fontsize=14, weight="bold")
ax.set_xlabel("Number of characters")
ax.set_ylabel("Count")
plt.show()

# 3. Query length (tokens) distribution
plt.figure(figsize=(10, 6))
ax = sns.histplot(df["query_len_tokens"], bins=30, kde=True, color="mediumseagreen")
ax.set_title("Query Text Length (Tokens)", fontsize=14, weight="bold")
ax.set_xlabel("Number of tokens")
ax.set_ylabel("Count")
plt.show()

# 4. Number of relevant windows per sample
plt.figure(figsize=(10, 6))
ax = sns.histplot(
    df["num_windows"], bins=range(1, df["num_windows"].max() + 2), color="mediumpurple"
)
ax.set_title("Relevant Windows per Sample", fontsize=14, weight="bold")
ax.set_xlabel("Number of windows")
ax.set_ylabel("Count")
plt.show()

In [ ]:
# 5. Interactive scatter: duration vs. number of windows (Plotly)
fig = px.scatter(
    df,
    x="duration",
    y="num_windows",
    hover_data=["qid", "vid"],
    title="Duration vs. Number of Relevant Windows",
    labels={"duration": "Duration (s)", "num_windows": "# Windows"},
    color="num_windows",
    color_continuous_scale=px.colors.sequential.Viridis,
)
fig.update_layout(template="plotly_dark")
# fig.write_html("duration_vs_windows.html")
fig.show()

In [ ]:
train_json = []

for data in dcase2026_clotho_train_json:
    data["vid"] = data["vid"].replace(".", "")
    train_json.append(data)

len(train_json)

In [ ]:
training_samples = []

for data in train_json:
    _vid = data["vid"] + ".wav"
    _local_path = str(TRAIN_DATASET_DIR / _vid)

    for _path, _ in train_pairs:
        if _local_path == _path:
            data["local_path"] = str(_local_path)
            training_samples.append(data)

len(training_samples)

In [ ]:
data_sample = training_samples[0]
data_sample

In [ ]:
def print_metadata(sample):
    print("\n=== Sample Metadata ===")
    for k, v in sample.items():
        # Skip the huge audio array if present
        if k == "local_path":
            print(f"{k:>15}: {v}")
        elif k == "relevant_windows":
            print(f"{k:>15}: {v}")
        else:
            print(f"{k:>15}: {v}")


print_metadata(data_sample)

### Explore Audio attributes

In [ ]:
y, sr = librosa.load(data_sample["local_path"])

In [ ]:
display(Audio(data=y, rate=sr))

In [ ]:
plt.figure()
librosa.display.waveshow(y, sr=sr, alpha=0.7)
plt.title("Waveform – qid 21288", fontsize=14, weight="bold")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

In [ ]:
# Parameters – you can tweak n_fft / hop_length for finer resolution
n_fft = 1024
hop_length = 256
D = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
plt.figure()
librosa.display.specshow(
    librosa.amplitude_to_db(D, ref=np.max),
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="log",
    cmap="magma",
)
plt.title("STFT Magnitude Spectrogram", fontsize=14, weight="bold")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

In [ ]:
mel_spec = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=40,
    fmax=sr / 2,
)
log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
plt.figure()
librosa.display.specshow(
    log_mel_spec,
    sr=sr,
    hop_length=hop_length,
    x_axis="time",
    y_axis="mel",
    cmap="viridis",
)
plt.title("Log‑Mel Spectrogram", fontsize=14, weight="bold")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Computation on the Dataset

In [ ]:
def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def span_xx_to_cxw(xx_spans):
    """
    Args:
        xx_spans: tensor, (#windows, 2) or (..., 2), each row is a window of format (st, ed)

    Returns:
        cxw_spans: tensor, (#windows, 2), each row is a window of format (center=(st+ed)/2, width=(ed-st))
    >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
    >>> span_xx_to_cxw(spans)
    tensor([[0.5000, 1.0000],
        [0.3000, 0.2000]])
    >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
    >>> span_xx_to_cxw(spans)
    tensor([[[0.5000, 1.0000],
         [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)

In [ ]:
# Init CLAP Models

In [ ]:
# Init Text-Embedding

In [ ]:
class StartEndDataset(Dataset):
    """One line in data loaded from data_path."
    {
      "qid": 7803,
      "query": "Man in gray top walks from outside to inside.",
      "duration": 150,
      "vid": "RoripwjYFp8_360.0_510.0",
      "relevant_clip_ids": [13, 14, 15, 16, 17],
      "relevant_windows": [[26, 36]]
    }
    """

    def __init__(
        self,
        data_path,
        a_feat_dir,
        q_feat_dir,
        q_feat_type="last_hidden_state",
        a_feat_type="pann",
        max_q_l=32,
        max_a_l=75,
        ctx_mode="video",
        clip_len=2,
        max_windows=5,
        span_loss_type="l1",
        load_labels=True,
    ):
        self.data_path = data_path
        self.a_feat_dir = a_feat_dir
        self.q_feat_dir = q_feat_dir
        self.q_feat_type = q_feat_type
        self.a_feat_type = a_feat_type

        if max_a_l == -1:
            max_a_l = 100000000

        if max_q_l == -1:
            max_q_l = 100

        self.max_q_l = max_q_l
        self.max_a_l = max_a_l

        self.ctx_mode = ctx_mode
        self.use_tef = "tef" in ctx_mode
        self.use_audio = "audio" in ctx_mode
        self.clip_len = clip_len
        self.max_windows = max_windows  # maximum number of windows to use as labels
        self.span_loss_type = span_loss_type
        self.load_labels = load_labels
        self.data = self.load_data()

    def load_data(self):
        datalist = load_jsonl(self.data_path)
        return datalist

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        meta = self.data[index]

        model_inputs = dict()
        model_inputs["query_feat"] = self._get_query_feat_by_qid(
            meta["qid"]
        )  # (Dq, ) or (Lq, Dq)
        model_inputs["audio_feat"] = self._get_audio_feat_by_vid(meta["vid"])
        ctx_l = len(model_inputs["audio_feat"])

        if self.use_tef:
            tef_st = torch.arange(0, ctx_l, 1.0) / ctx_l
            tef_ed = tef_st + 1.0 / ctx_l
            tef = torch.stack([tef_st, tef_ed], dim=1)  # (Lv, 2)
            model_inputs["audio_feat"] = torch.cat(
                [model_inputs["audio_feat"], tef], dim=1
            )

        if self.load_labels:
            model_inputs["span_labels"] = self.get_span_labels(
                meta["relevant_windows"], ctx_l
            )
            (
                model_inputs["saliency_pos_labels"],
                model_inputs["saliency_neg_labels"],
                model_inputs["saliency_all_labels"],
            ) = self.get_saliency_labels_sub_as_query(
                meta["relevant_windows"][0], ctx_l
            )

        return dict(meta=meta, model_inputs=model_inputs)

    def get_pos_mask(self, meta, ctx_l):
        # necessary only for TR-DETR: model_inputs["pos_mask"]
        if "relevant_clip_ids" in meta:
            pos_idx = torch.tensor(meta["relevant_clip_ids"])
        else:
            # TODO: Implemented pos_mask for MR/HD tasks for TR-DETR, but I could not reproduce the reported scores
            clip_start_ind = math.floor(meta["relevant_windows"][0][0] / self.clip_len)
            clip_end_ind = math.ceil(meta["relevant_windows"][0][1] / self.clip_len)
            if clip_start_ind == clip_end_ind:
                clip_end_ind += 1  # to avoid a bug
            pos_idx = torch.tensor([i for i in range(clip_start_ind, clip_end_ind)])

        mask = torch.zeros_like(torch.ones(ctx_l))
        if pos_idx.max() >= len(mask):
            new_mask = torch.zeros_like(torch.ones(pos_idx.max() + 1))
            new_mask[pos_idx] = 1
            new_mask[: len(mask)] = mask
            mask = new_mask
        else:
            mask[pos_idx] = 1

        if self.dset_name in [
            "charades",
            "tacos",
            "activitynet",
            "clotho-moment",
            "unav100-subset",
            "tut2017",
        ]:
            mask = mask[:ctx_l]

        return mask

    def get_saliency_labels_sub_as_query(self, gt_window, ctx_l, max_n=2):
        gt_st = int(gt_window[0] / self.clip_len)
        gt_ed = max(0, min(int(gt_window[1] / self.clip_len), ctx_l) - 1)

        if gt_st > gt_ed:
            gt_st = gt_ed

        if gt_st != gt_ed:
            pos_clip_indices = random.sample(range(gt_st, gt_ed + 1), k=max_n)
        else:
            pos_clip_indices = [gt_st, gt_st]

        neg_pool = list(range(0, gt_st)) + list(
            range(gt_ed + 1, ctx_l)
        )  # to fix bugs / works..?
        try:
            neg_clip_indices = random.sample(neg_pool, k=max_n)
        except:
            neg_clip_indices = pos_clip_indices

        score_array = np.zeros(ctx_l)
        score_array[gt_st : gt_ed + 1] = 1

        return pos_clip_indices, neg_clip_indices, score_array

    def get_saliency_labels(
        self, rel_clip_ids, scores, ctx_l, max_n=1, add_easy_negative=True
    ):
        """Sum the scores from the three annotations, then take the two clips with the
        maximum scores as positive, and two with the minimum scores as negative.
        Args:
            rel_clip_ids: list(int), list of relevant clip ids
            scores: list([anno1_score, anno2_score, anno3_score]),
            ctx_l: int
            max_n: int, #clips to use as positive and negative, for easy and hard negative, respectively.
            add_easy_negative: bool, if True, sample eay negative outside the relevant_clip_ids.
        """
        # indices inside rel_clip_ids
        scores = np.array(scores)  # (#rel_clips, 3)
        agg_scores = np.sum(scores, 1)  # (#rel_clips, )
        sort_indices = np.argsort(agg_scores)  # increasing

        # indices in the whole video
        # the min(_, ctx_l-1) here is incorrect, but should not cause
        # much troubles since this should be rarely used.
        hard_pos_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[-max_n:]
        ]
        hard_neg_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[:max_n]
        ]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)) - set(rel_clip_ids))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices
        return pos_clip_indices, neg_clip_indices

    def get_saliency_labels_all(
        self, rel_clip_ids, scores, ctx_l, max_n=1, add_easy_negative=True
    ):
        """Sum the scores from the three annotations, then take the two clips with the
        maximum scores as positive, and two with the minimum scores as negative.
        Args:
            rel_clip_ids: list(int), list of relevant clip ids
            scores: list([anno1_score, anno2_score, anno3_score]),
            ctx_l: int
            max_n: int, #clips to use as positive and negative, for easy and hard negative, respectively.
            add_easy_negative: bool, if True, sample eay negative outside the relevant_clip_ids.
        """
        # indices inside rel_clip_ids
        scores = np.array(scores)  # (#rel_clips, 3)
        agg_scores = np.sum(scores, 1)  # (#rel_clips, )
        sort_indices = np.argsort(agg_scores)  # increasing

        # score_array = [min(agg_scores[idx], ctx_l-1) for idx in range(ctx_l)]
        score_array = np.zeros(ctx_l)
        for idx in range(len(rel_clip_ids)):
            if rel_clip_ids[idx] >= ctx_l:
                score_array_new = np.zeros(ctx_l + 1)
                score_array_new[:ctx_l] = score_array
                score_array = score_array_new
            # if rel_clip_ids[idx] == ctx_l:
            #     print(rel_clip_ids[idx], ctx_l)
            score_array[rel_clip_ids[idx]] = agg_scores[idx]

        # indices in the whole video
        # the min(_, ctx_l-1) here is incorrect, but should not cause
        # much troubles since this should be rarely used.
        hard_pos_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[-max_n:]
        ]
        hard_neg_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[:max_n]
        ]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)) - set(rel_clip_ids))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices
        return pos_clip_indices, neg_clip_indices, score_array

    def get_saliency_labels_all_tvsum(
        self, labels, ctx_l, max_n=1, add_easy_negative=False
    ):

        agg_scores = np.sum(labels - np.ones_like(labels), axis=-1)[
            :ctx_l
        ]  # start from 1, so minus 1
        score_array = agg_scores / 80 * 12
        sort_indices = np.argsort(agg_scores)  # increasing

        hard_pos_clip_indices = [min(idx, ctx_l - 1) for idx in sort_indices[-max_n:]]
        hard_neg_clip_indices = [min(idx, ctx_l - 1) for idx in sort_indices[:max_n]]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices

        return pos_clip_indices, neg_clip_indices, score_array

    def get_saliency_labels_all_youtube(
        self, labels, ctx_l, max_n=1, add_easy_negative=False
    ):
        # Youtube-hl only have binary score
        agg_scores = np.array(labels)[:, 0]  # (L, 1) --> (L, )
        score_array = agg_scores * 1

        sort_indices = np.argsort(agg_scores)  # increasing

        hard_pos_clip_indices = [min(idx, ctx_l - 1) for idx in sort_indices[-max_n:]]
        hard_neg_clip_indices = [min(idx, ctx_l - 1) for idx in sort_indices[:max_n]]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices

        return pos_clip_indices, neg_clip_indices, score_array

    def get_span_labels(self, windows, ctx_l):
        """
        windows: list([st, ed]) in seconds. E.g. [[26, 36]], corresponding st_ed clip_indices [[13, 17]] (inclusive)
            Note a maximum of `self.max_windows` windows are used.
        returns Tensor of shape (#windows, 2), each row is [center, width] normalized by video length
        """
        if len(windows) > self.max_windows:
            random.shuffle(windows)
            windows = windows[: self.max_windows]
        if self.span_loss_type == "l1":
            windows = torch.Tensor(windows) / (
                ctx_l * self.clip_len
            )  # normalized windows in xx
            windows = span_xx_to_cxw(windows)  # normalized windows in cxw
        elif self.span_loss_type == "ce":
            windows = torch.Tensor(
                [
                    [
                        int(w[0] / self.clip_len),
                        min(int(w[1] / self.clip_len), ctx_l) - 1,
                    ]
                    for w in windows
                ]
            ).long()  # inclusive
        else:
            raise NotImplementedError
        return windows

    # NOTE: This is changable since this is processed via an embedding mdoels
    def _get_query_feat_by_qid(self, qid):
        q_feat_path = join(self.q_feat_dir, f"qid{qid}.npz")
        q_feat = np.load(q_feat_path)["last_hidden_state"]
        return q_feat

    # NOTE: This is changable since this is processed via an embedding mdoels
    def _get_audio_feat_by_vid(self, vid):
        _feat_path = join(self.a_feat_dir, f"{vid}.npz")
        _feat = np.load(_feat_path)["features"][: self.max_a_l].astype(np.float32)
        _feat = l2_normalize_np_array(_feat)
        return torch.from_numpy(_feat)